# 05 · Transcripción contextual con Whisper

Transcribe cada audio con `faster-whisper` y alinea las palabras con los segmentos diarizados consolidados en el Notebook 04.

El notebook orquesta la fase en celdas visibles (igual que el Notebook 01): construir contexto, restaurar desde GCS, cargar segmentos, resolver audios, auditar lo ya hecho, cargar modelo, **bucle de transcripción con checkpoints**, consolidar y publicar.

**Reanudación en dos niveles:**
- *Salto de fase completa:* si los outputs finales ya existen y `FORCE_TRANSCRIPTION=False`, no se vuelve a ejecutar Whisper.
- *Checkpoint por audio:* dentro del bucle, cada audio válido se omite y cada `N` audios se guarda y sube el progreso a GCS.

In [1]:
# INSTALACIÓN DE REQUISITOS PREVIOS
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")

Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS Y ACCESO AL REPOSITORIO
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from google.cloud import storage
from IPython.display import display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    GCS_CLEAN_AUDIO_PREFIX,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    INPUT_DIR,
    OUTPUT_DIR,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
)
from src.storage_io import download_prefix_to_local, ensure_local_file
from src import transcripcion_contextual as tc

gcs_client = storage.Client()
print("Repositorio y cliente GCS configurados.")

Repositorio y cliente GCS configurados.


In [3]:
# CONFIGURACIÓN DE LA FASE 05
# Todos los parámetros ajustables viven aquí. Cambia cualquier número
# en esta celda sin tocar los archivos .py.

# --- Control de ejecución ---
FORCE_TRANSCRIPTION = False        # True = reejecuta aunque existan outputs finales
MAX_AUDIOS_TO_PROCESS = None       # None = todos; un número limita la corrida
UPLOAD_EACH_AUDIO_TO_GCS = True    # sube outputs por audio al terminar cada uno

# --- Modelo Whisper ---
MODEL_SIZE = "large-v3-turbo"
LANGUAGE = "es"
TARGET_SR = 16000

INITIAL_PROMPT = (
    "Conversación telefónica en español entre un agente y un cliente. "
    "Transcripción literal con números, fechas, nombres y términos comerciales."
)
HOTWORDS = None                    # términos del dominio separados por comas, o None
USE_VAD = False                    # audios ya limpios; VAD desactivado evita vacíos

TRANSCRIBE_OPTIONS = {
    "language": LANGUAGE,
    "task": "transcribe",
    "beam_size": 1,
    "temperature": 0.0,
    "word_timestamps": True,
    "without_timestamps": False,
    "vad_filter": USE_VAD,
    "condition_on_previous_text": True,
    "prompt_reset_on_temperature": 0.5,
    "initial_prompt": INITIAL_PROMPT,
    "hotwords": HOTWORDS,
    "repetition_penalty": 1.05,
    "no_repeat_ngram_size": 3,
    "log_progress": False,
}

# --- Reintentos para audios de baja energía ---
RETRY_EMPTY_WITH_AUDIO_GAIN = True
MAX_RETRY_GAIN = 10.0
TARGET_RETRY_RMS = 0.08

# --- Alineación y checkpoints ---
NEAREST_WORD_TOLERANCE_SEC = 0.40  # tolerancia palabra->segmento más cercano
LOW_WORD_PROBABILITY = 0.50        # umbral para contar palabras de baja confianza
SAVE_CHECKPOINT_EVERY_N_AUDIOS = 25
EXPECTED_FINAL_AUDIO_IDS = 1181    # nº de audios esperado (solo para aviso)

print("Modelo:", MODEL_SIZE, "| idioma:", LANGUAGE, "| VAD:", USE_VAD)
print("Forzar transcripción:", FORCE_TRANSCRIPTION)
print("Checkpoint cada N audios:", SAVE_CHECKPOINT_EVERY_N_AUDIOS)

Modelo: large-v3-turbo | idioma: es | VAD: False
Forzar transcripción: False
Checkpoint cada N audios: 25


In [4]:
# RESTAURACIÓN DESDE GCS Y SALTO DE FASE COMPLETA
# Restaura los outputs de la fase desde GCS. Si ya están completos y no se
# fuerza, se reutilizan sin volver a ejecutar Whisper.

tc.restore_phase_outputs_from_gcs(gcs_client)

SKIP_PHASE = tc.phase_outputs_complete() and not FORCE_TRANSCRIPTION

if SKIP_PHASE:
    phase_outputs = tc.load_existing_phase_outputs()
    print("Outputs completos restaurados. No se vuelve a ejecutar Whisper.")
    print("Segmentos:", len(phase_outputs["df_transcribed"]))
else:
    print("Se ejecutará la transcripción (o se completarán los audios pendientes).")

Restauración desde GCS completada.
Archivos encontrados: 10701
Archivos descargados: 0
Archivos locales ya vigentes: 10701
Outputs completos restaurados. No se vuelve a ejecutar Whisper.
Segmentos: 40352


In [5]:
# CONTEXTO DE EJECUCIÓN Y CARGA DE SEGMENTOS DIARIZADOS

if not SKIP_PHASE:
    ctx = tc.build_transcription_context(
        gcs_client,
        model_size=MODEL_SIZE,
        language=LANGUAGE,
        target_sr=TARGET_SR,
        transcribe_options=TRANSCRIBE_OPTIONS,
        retry_empty_with_audio_gain=RETRY_EMPTY_WITH_AUDIO_GAIN,
        max_retry_gain=MAX_RETRY_GAIN,
        target_retry_rms=TARGET_RETRY_RMS,
        save_checkpoint_every_n=SAVE_CHECKPOINT_EVERY_N_AUDIOS,
        nearest_word_tolerance_sec=NEAREST_WORD_TOLERANCE_SEC,
        low_word_probability=LOW_WORD_PROBABILITY,
        expected_final_audio_ids=EXPECTED_FINAL_AUDIO_IDS,
    )
    print("Run ID:", ctx.run_id)

    ensure_local_file(
        CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
        gcs_client,
        gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
        base_dir=OUTPUT_DIR,
        required=True,
    )
    download_prefix_to_local(
        source_uri=GCS_CLEAN_AUDIO_PREFIX,
        local_dir=INPUT_DIR,
        gcs_client=gcs_client,
        suffix=".wav",
        force=False,
    )

    (df_segments, target_audio_ids,
     audio_fingerprints, expected_counts) = tc.load_and_validate_segments()

    print("Segmentos finales:", len(df_segments))
    print("Audios únicos:", len(target_audio_ids))
    if len(target_audio_ids) != EXPECTED_FINAL_AUDIO_IDS:
        print(f"Aviso: se esperaban {EXPECTED_FINAL_AUDIO_IDS} audios.")

In [6]:
# RESOLUCIÓN DE AUDIOS LIMPIOS Y RESTAURACIÓN DEL RUN

if not SKIP_PHASE:
    df_audio_catalog, audio_paths, df_missing = tc.build_audio_catalog(
        ctx, df_segments, audio_fingerprints
    )
    print("Audios localizados:", int(df_audio_catalog["audio_exists"].sum()))
    print("Audios no localizados:", len(df_missing))

    if len(df_missing):
        display(df_missing.head(30))
        raise FileNotFoundError("Faltan audios limpios. Se detiene antes de cargar el modelo.")

    restored = tc.restore_run_from_gcs(ctx)
    print("Archivos del run restaurados desde GCS:", restored)

In [7]:
# AUDITORÍA DE OUTPUTS PREVIOS (checkpoint: qué está hecho y qué falta)

if not SKIP_PHASE:
    df_audit, valid_ids, pending_ids = tc.audit_and_clean_run_outputs(
        ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints
    )
    print("Audios ya válidos:", len(valid_ids))
    print("Audios pendientes:", len(pending_ids))
    if not df_audit.empty:
        display(df_audit["reason"].value_counts().rename_axis("reason").reset_index(name="n"))

In [8]:
# CARGA DEL MODELO WHISPER

if not SKIP_PHASE and pending_ids:
    ctx = tc.load_whisper_model(ctx)
elif not SKIP_PHASE:
    print("No hay audios pendientes; no se carga el modelo.")

In [9]:
# BUCLE DE TRANSCRIPCIÓN CON CHECKPOINTS POR AUDIO
# Cada audio se transcribe, valida y sube a GCS. Cada N audios se guarda el
# manifiesto de progreso (checkpoint) y se sube a GCS, de modo que una
# interrupción se reanuda sin reprocesar lo ya hecho.

if not SKIP_PHASE:
    to_process = list(pending_ids)
    if MAX_AUDIOS_TO_PROCESS is not None:
        to_process = to_process[: int(MAX_AUDIOS_TO_PROCESS)]

    print(f"Audios a procesar en esta ejecución: {len(to_process)}")
    batch_updates = []

    for batch_index, audio_id_base in enumerate(to_process, start=1):
        global_position = len(valid_ids) + batch_index
        print(f"[{global_position}/{len(target_audio_ids)}] Transcribiendo {audio_id_base}...", flush=True)

        try:
            _, _, _, _, summary = tc.process_one_audio(
                ctx, audio_id_base, df_segments, audio_paths, audio_fingerprints,
                expected_counts=expected_counts, save_outputs=True,
                upload_each_audio=UPLOAD_EACH_AUDIO_TO_GCS,
            )
            batch_updates.append(summary)
            print(f"  OK | {summary['n_words_total']} palabras | {summary['elapsed_sec']} s")
        except Exception as exc:
            batch_updates.append({
                "audio_id_base": audio_id_base, "status": "error", "error": str(exc),
                "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            print("  ERROR:", exc)

        if (batch_index % SAVE_CHECKPOINT_EVERY_N_AUDIOS == 0
                or batch_index == len(to_process)):
            tc.save_processing_checkpoint(
                ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints,
                processed_counter=global_position, extra_rows=batch_updates,
            )
            print("  Checkpoint local y GCS guardado.")

    print("Batch finalizado.")

In [10]:
# VALIDACIÓN FINAL, CONSOLIDACIÓN Y PUBLICACIÓN CANÓNICA

if not SKIP_PHASE:
    run_complete, df_transcribed, df_summary, df_final_audit = tc.finalize_and_consolidate(
        ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints
    )
    print("Outputs válidos:", int(df_final_audit["valid"].sum()))
    print("Faltantes o inválidos:", int((~df_final_audit["valid"]).sum()))

    if run_complete:
        active_run = tc.publish_local_canonical(ctx, target_audio_ids, df_transcribed)
        tc.publish_gcs_canonical(ctx)
        print("\nPublicación final completada:")
        print(active_run)
        phase_outputs = {
            "df_transcribed": df_transcribed,
            "df_transcription_summary": df_summary,
            "active_run": active_run,
            "reused": False,
        }
    else:
        display(df_final_audit.loc[~df_final_audit["valid"]].head(50))
        print("La corrida permanece en staging y NO sustituye los outputs activos.")

In [11]:
# ASEGURAR SUBIDA A GCS (INCONDICIONAL)
# Se ejecuta siempre, se haya corrido la fase o se haya saltado por tener
# outputs locales completos. Evita el caso 'lo corrí en local pero nunca
# se subió al bucket'. skip_unchanged hace que sea barato si ya estaba.

tc.ensure_canonical_outputs_in_gcs(gcs_client)

Subiendo outputs 4744/10703: runs/full_audio_word_alignment_v2_large_v3_88f29fd88d31/run_config.json
Subiendo outputs 10702/10703: tabla_resumen_transcripcion_segmentada.csv
Subida final completada.
Archivos locales revisados: 10703
Archivos subidos: 2
Archivos omitidos sin cambios: 10701
Destino: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/
Sincronización con GCS asegurada (incondicional).


{'total': 10703, 'uploaded': 2, 'skipped': 10701}

In [12]:
# CONTROL FINAL DE CALIDAD

if "phase_outputs" in dir() and phase_outputs.get("df_transcribed") is not None:
    df_active = phase_outputs["df_transcribed"]
    con_texto = int(df_active["text"].fillna("").astype(str).str.strip().ne("").sum())
    print("Audios únicos:", df_active["audio_id_base"].nunique())
    print("Segmentos:", len(df_active))
    print("Segmentos con texto:", con_texto)
    print("Cobertura de texto: {:.1%}".format(con_texto / len(df_active) if len(df_active) else 0))

    sample_cols = [c for c in ["audio_file", "start", "end", "speaker_final", "text",
                               "n_words_assigned", "transcription_status"] if c in df_active.columns]
    con_texto_df = df_active.loc[df_active["text"].fillna("").astype(str).str.strip().ne("")]
    if len(con_texto_df):
        display(con_texto_df[sample_cols].sample(min(20, len(con_texto_df)), random_state=42)
                .sort_values(["audio_file", "start"]))
else:
    print("No hay outputs activos publicados en esta ejecución.")

Audios únicos: 1181
Segmentos: 40352
Segmentos con texto: 38872
Cobertura de texto: 96.3%


,audio_file,start,end,speaker_final,text,n_words_assigned,transcription_status
1119,raw_9154487929260056851_clean.wav,182.804094,187.799094,SPEAKER_00,"Vale, porque he visto desde mi página que sí que sale el 14C, te lo digo a ver.",18,ok
4117,raw_9155938668110006851_clean.wav,131.487219,149.999094,SPEAKER_00,"Pues muchísimas gracias. Aquí, resumiendo el motivo de tu llamada, querías saber cómo contactar con Legalitess si lo tenías incluido y hemos visto que sí. Y te ha facilitado el...",63,ok
6502,raw_9156095001490006851_clean.wav,84.726594,88.034094,SPEAKER_01,le deas para que no nos pisemos y se lo volvamos a activar.,13,ok
8876,raw_bajas_9156251225660016851_clean.wav,70.855344,73.234719,SPEAKER_01,"Sí, exacto. Y quiero quitar una de las cuatro línias.",10,ok
10987,raw_bajas_9156329881470026851_clean.wav,8.502219,9.244719,SPEAKER_00,Con Nervi.,2,ok
11309,raw_bajas_9156336807940016851_clean.wav,93.062844,95.644719,SPEAKER_01,"Bueno, no sé. Te quedan agosto cuando tal, que pasaba.",10,ok
11997,raw_bajas_9156372134020006851_clean.wav,41.695344,43.838469,SPEAKER_00,A ver cómo miro esto.,5,ok
14254,raw_bajas_9156485045910016851_clean.wav,0.554094,4.266594,SPEAKER_00,"Buenos días, bienvenido a Yaceli a la Cisina del Área de Baja. ¿Te puedo ayudar?",15,ok
17099,raw_bajas_9156563766870016851_clean.wav,149.155344,157.424094,SPEAKER_01,"Vale, comprendo. Y bueno, cuéntame, cuéntame, Estefan. Te vas al domicilio donde te vas a mudar. ¿Ya tienes servicio de internet?",21,ok
18743,raw_bajas_9156641416650006851_clean.wav,107.963469,108.739719,SPEAKER_00,No le funcionan. No,4,ok


# 05 · Transcripción contextual con Whisper

Transcribe cada audio con `faster-whisper` y alinea las palabras con los segmentos diarizados consolidados en el Notebook 04.

El notebook orquesta la fase en celdas visibles (igual que el Notebook 01): construir contexto, restaurar desde GCS, cargar segmentos, resolver audios, auditar lo ya hecho, cargar modelo, **bucle de transcripción con checkpoints**, consolidar y publicar.

**Reanudación en dos niveles:**
- *Salto de fase completa:* si los outputs finales ya existen y `FORCE_TRANSCRIPTION=False`, no se vuelve a ejecutar Whisper.
- *Checkpoint por audio:* dentro del bucle, cada audio válido se omite y cada `N` audios se guarda y sube el progreso a GCS.

In [1]:
# INSTALACIÓN DE REQUISITOS PREVIOS
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")

Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS Y ACCESO AL REPOSITORIO
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from google.cloud import storage
from IPython.display import display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    GCS_CLEAN_AUDIO_PREFIX,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    INPUT_DIR,
    OUTPUT_DIR,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
)
from src.storage_io import download_prefix_to_local, ensure_local_file
from src import transcripcion_contextual as tc

gcs_client = storage.Client()
print("Repositorio y cliente GCS configurados.")

Repositorio y cliente GCS configurados.


In [3]:
# CONFIGURACIÓN DE LA FASE 05
# Todos los parámetros ajustables viven aquí. Cambia cualquier número
# en esta celda sin tocar los archivos .py.

# --- Control de ejecución ---
FORCE_TRANSCRIPTION = False        # True = reejecuta aunque existan outputs finales
MAX_AUDIOS_TO_PROCESS = None       # None = todos; un número limita la corrida
UPLOAD_EACH_AUDIO_TO_GCS = True    # sube outputs por audio al terminar cada uno

# --- Modelo Whisper ---
MODEL_SIZE = "large-v3-turbo"
LANGUAGE = "es"
TARGET_SR = 16000

INITIAL_PROMPT = (
    "Conversación telefónica en español entre un agente y un cliente. "
    "Transcripción literal con números, fechas, nombres y términos comerciales."
)
HOTWORDS = None                    # términos del dominio separados por comas, o None
USE_VAD = False                    # audios ya limpios; VAD desactivado evita vacíos

TRANSCRIBE_OPTIONS = {
    "language": LANGUAGE,
    "task": "transcribe",
    "beam_size": 1,
    "temperature": 0.0,
    "word_timestamps": True,
    "without_timestamps": False,
    "vad_filter": USE_VAD,
    "condition_on_previous_text": True,
    "prompt_reset_on_temperature": 0.5,
    "initial_prompt": INITIAL_PROMPT,
    "hotwords": HOTWORDS,
    "repetition_penalty": 1.05,
    "no_repeat_ngram_size": 3,
    "log_progress": False,
}

# --- Reintentos para audios de baja energía ---
RETRY_EMPTY_WITH_AUDIO_GAIN = True
MAX_RETRY_GAIN = 10.0
TARGET_RETRY_RMS = 0.08

# --- Alineación y checkpoints ---
NEAREST_WORD_TOLERANCE_SEC = 0.40  # tolerancia palabra->segmento más cercano
LOW_WORD_PROBABILITY = 0.50        # umbral para contar palabras de baja confianza
SAVE_CHECKPOINT_EVERY_N_AUDIOS = 25
EXPECTED_FINAL_AUDIO_IDS = 1181    # nº de audios esperado (solo para aviso)

print("Modelo:", MODEL_SIZE, "| idioma:", LANGUAGE, "| VAD:", USE_VAD)
print("Forzar transcripción:", FORCE_TRANSCRIPTION)
print("Checkpoint cada N audios:", SAVE_CHECKPOINT_EVERY_N_AUDIOS)

Modelo: large-v3-turbo | idioma: es | VAD: False
Forzar transcripción: False
Checkpoint cada N audios: 25


In [4]:
# RESTAURACIÓN DESDE GCS Y SALTO DE FASE COMPLETA
# Restaura los outputs de la fase desde GCS. Si ya están completos y no se
# fuerza, se reutilizan sin volver a ejecutar Whisper.

tc.restore_phase_outputs_from_gcs(gcs_client)

SKIP_PHASE = tc.phase_outputs_complete() and not FORCE_TRANSCRIPTION

if SKIP_PHASE:
    phase_outputs = tc.load_existing_phase_outputs()
    print("Outputs completos restaurados. No se vuelve a ejecutar Whisper.")
    print("Segmentos:", len(phase_outputs["df_transcribed"]))
else:
    print("Se ejecutará la transcripción (o se completarán los audios pendientes).")

Restauración desde GCS completada.
Archivos encontrados: 10701
Archivos descargados: 0
Archivos locales ya vigentes: 10701
Outputs completos restaurados. No se vuelve a ejecutar Whisper.
Segmentos: 40352


In [5]:
# CONTEXTO DE EJECUCIÓN Y CARGA DE SEGMENTOS DIARIZADOS

if not SKIP_PHASE:
    ctx = tc.build_transcription_context(
        gcs_client,
        model_size=MODEL_SIZE,
        language=LANGUAGE,
        target_sr=TARGET_SR,
        transcribe_options=TRANSCRIBE_OPTIONS,
        retry_empty_with_audio_gain=RETRY_EMPTY_WITH_AUDIO_GAIN,
        max_retry_gain=MAX_RETRY_GAIN,
        target_retry_rms=TARGET_RETRY_RMS,
        save_checkpoint_every_n=SAVE_CHECKPOINT_EVERY_N_AUDIOS,
        nearest_word_tolerance_sec=NEAREST_WORD_TOLERANCE_SEC,
        low_word_probability=LOW_WORD_PROBABILITY,
        expected_final_audio_ids=EXPECTED_FINAL_AUDIO_IDS,
    )
    print("Run ID:", ctx.run_id)

    ensure_local_file(
        CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
        gcs_client,
        gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
        base_dir=OUTPUT_DIR,
        required=True,
    )
    download_prefix_to_local(
        source_uri=GCS_CLEAN_AUDIO_PREFIX,
        local_dir=INPUT_DIR,
        gcs_client=gcs_client,
        suffix=".wav",
        force=False,
    )

    (df_segments, target_audio_ids,
     audio_fingerprints, expected_counts) = tc.load_and_validate_segments()

    print("Segmentos finales:", len(df_segments))
    print("Audios únicos:", len(target_audio_ids))
    if len(target_audio_ids) != EXPECTED_FINAL_AUDIO_IDS:
        print(f"Aviso: se esperaban {EXPECTED_FINAL_AUDIO_IDS} audios.")

In [6]:
# RESOLUCIÓN DE AUDIOS LIMPIOS Y RESTAURACIÓN DEL RUN

if not SKIP_PHASE:
    df_audio_catalog, audio_paths, df_missing = tc.build_audio_catalog(
        ctx, df_segments, audio_fingerprints
    )
    print("Audios localizados:", int(df_audio_catalog["audio_exists"].sum()))
    print("Audios no localizados:", len(df_missing))

    if len(df_missing):
        display(df_missing.head(30))
        raise FileNotFoundError("Faltan audios limpios. Se detiene antes de cargar el modelo.")

    restored = tc.restore_run_from_gcs(ctx)
    print("Archivos del run restaurados desde GCS:", restored)

In [7]:
# AUDITORÍA DE OUTPUTS PREVIOS (checkpoint: qué está hecho y qué falta)

if not SKIP_PHASE:
    df_audit, valid_ids, pending_ids = tc.audit_and_clean_run_outputs(
        ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints
    )
    print("Audios ya válidos:", len(valid_ids))
    print("Audios pendientes:", len(pending_ids))
    if not df_audit.empty:
        display(df_audit["reason"].value_counts().rename_axis("reason").reset_index(name="n"))

In [8]:
# CARGA DEL MODELO WHISPER

if not SKIP_PHASE and pending_ids:
    ctx = tc.load_whisper_model(ctx)
elif not SKIP_PHASE:
    print("No hay audios pendientes; no se carga el modelo.")

In [9]:
# BUCLE DE TRANSCRIPCIÓN CON CHECKPOINTS POR AUDIO
# Cada audio se transcribe, valida y sube a GCS. Cada N audios se guarda el
# manifiesto de progreso (checkpoint) y se sube a GCS, de modo que una
# interrupción se reanuda sin reprocesar lo ya hecho.

if not SKIP_PHASE:
    to_process = list(pending_ids)
    if MAX_AUDIOS_TO_PROCESS is not None:
        to_process = to_process[: int(MAX_AUDIOS_TO_PROCESS)]

    print(f"Audios a procesar en esta ejecución: {len(to_process)}")
    batch_updates = []

    for batch_index, audio_id_base in enumerate(to_process, start=1):
        global_position = len(valid_ids) + batch_index
        print(f"[{global_position}/{len(target_audio_ids)}] Transcribiendo {audio_id_base}...", flush=True)

        try:
            _, _, _, _, summary = tc.process_one_audio(
                ctx, audio_id_base, df_segments, audio_paths, audio_fingerprints,
                expected_counts=expected_counts, save_outputs=True,
                upload_each_audio=UPLOAD_EACH_AUDIO_TO_GCS,
            )
            batch_updates.append(summary)
            print(f"  OK | {summary['n_words_total']} palabras | {summary['elapsed_sec']} s")
        except Exception as exc:
            batch_updates.append({
                "audio_id_base": audio_id_base, "status": "error", "error": str(exc),
                "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            print("  ERROR:", exc)

        if (batch_index % SAVE_CHECKPOINT_EVERY_N_AUDIOS == 0
                or batch_index == len(to_process)):
            tc.save_processing_checkpoint(
                ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints,
                processed_counter=global_position, extra_rows=batch_updates,
            )
            print("  Checkpoint local y GCS guardado.")

    print("Batch finalizado.")

In [10]:
# VALIDACIÓN FINAL, CONSOLIDACIÓN Y PUBLICACIÓN CANÓNICA

if not SKIP_PHASE:
    run_complete, df_transcribed, df_summary, df_final_audit = tc.finalize_and_consolidate(
        ctx, target_audio_ids, df_segments, expected_counts, audio_fingerprints
    )
    print("Outputs válidos:", int(df_final_audit["valid"].sum()))
    print("Faltantes o inválidos:", int((~df_final_audit["valid"]).sum()))

    if run_complete:
        active_run = tc.publish_local_canonical(ctx, target_audio_ids, df_transcribed)
        tc.publish_gcs_canonical(ctx)
        print("\nPublicación final completada:")
        print(active_run)
        phase_outputs = {
            "df_transcribed": df_transcribed,
            "df_transcription_summary": df_summary,
            "active_run": active_run,
            "reused": False,
        }
    else:
        display(df_final_audit.loc[~df_final_audit["valid"]].head(50))
        print("La corrida permanece en staging y NO sustituye los outputs activos.")

In [11]:
# ASEGURAR SUBIDA A GCS (INCONDICIONAL)
# Se ejecuta siempre, se haya corrido la fase o se haya saltado por tener
# outputs locales completos. Evita el caso 'lo corrí en local pero nunca
# se subió al bucket'. skip_unchanged hace que sea barato si ya estaba.

tc.ensure_canonical_outputs_in_gcs(gcs_client)

Subiendo outputs 4744/10703: runs/full_audio_word_alignment_v2_large_v3_88f29fd88d31/run_config.json
Subiendo outputs 10702/10703: tabla_resumen_transcripcion_segmentada.csv
Subida final completada.
Archivos locales revisados: 10703
Archivos subidos: 2
Archivos omitidos sin cambios: 10701
Destino: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/
Sincronización con GCS asegurada (incondicional).


{'total': 10703, 'uploaded': 2, 'skipped': 10701}

In [12]:
# CONTROL FINAL DE CALIDAD

if "phase_outputs" in dir() and phase_outputs.get("df_transcribed") is not None:
    df_active = phase_outputs["df_transcribed"]
    con_texto = int(df_active["text"].fillna("").astype(str).str.strip().ne("").sum())
    print("Audios únicos:", df_active["audio_id_base"].nunique())
    print("Segmentos:", len(df_active))
    print("Segmentos con texto:", con_texto)
    print("Cobertura de texto: {:.1%}".format(con_texto / len(df_active) if len(df_active) else 0))

    sample_cols = [c for c in ["audio_file", "start", "end", "speaker_final", "text",
                               "n_words_assigned", "transcription_status"] if c in df_active.columns]
    con_texto_df = df_active.loc[df_active["text"].fillna("").astype(str).str.strip().ne("")]
    if len(con_texto_df):
        display(con_texto_df[sample_cols].sample(min(20, len(con_texto_df)), random_state=42)
                .sort_values(["audio_file", "start"]))
else:
    print("No hay outputs activos publicados en esta ejecución.")

Audios únicos: 1181
Segmentos: 40352
Segmentos con texto: 38872
Cobertura de texto: 96.3%


,audio_file,start,end,speaker_final,text,n_words_assigned,transcription_status
1119,raw_9154487929260056851_clean.wav,182.804094,187.799094,SPEAKER_00,"Vale, porque he visto desde mi página que sí que sale el 14C, te lo digo a ver.",18,ok
4117,raw_9155938668110006851_clean.wav,131.487219,149.999094,SPEAKER_00,"Pues muchísimas gracias. Aquí, resumiendo el motivo de tu llamada, querías saber cómo contactar con Legalitess si lo tenías incluido y hemos visto que sí. Y te ha facilitado el...",63,ok
6502,raw_9156095001490006851_clean.wav,84.726594,88.034094,SPEAKER_01,le deas para que no nos pisemos y se lo volvamos a activar.,13,ok
8876,raw_bajas_9156251225660016851_clean.wav,70.855344,73.234719,SPEAKER_01,"Sí, exacto. Y quiero quitar una de las cuatro línias.",10,ok
10987,raw_bajas_9156329881470026851_clean.wav,8.502219,9.244719,SPEAKER_00,Con Nervi.,2,ok
11309,raw_bajas_9156336807940016851_clean.wav,93.062844,95.644719,SPEAKER_01,"Bueno, no sé. Te quedan agosto cuando tal, que pasaba.",10,ok
11997,raw_bajas_9156372134020006851_clean.wav,41.695344,43.838469,SPEAKER_00,A ver cómo miro esto.,5,ok
14254,raw_bajas_9156485045910016851_clean.wav,0.554094,4.266594,SPEAKER_00,"Buenos días, bienvenido a Yaceli a la Cisina del Área de Baja. ¿Te puedo ayudar?",15,ok
17099,raw_bajas_9156563766870016851_clean.wav,149.155344,157.424094,SPEAKER_01,"Vale, comprendo. Y bueno, cuéntame, cuéntame, Estefan. Te vas al domicilio donde te vas a mudar. ¿Ya tienes servicio de internet?",21,ok
18743,raw_bajas_9156641416650006851_clean.wav,107.963469,108.739719,SPEAKER_00,No le funcionan. No,4,ok
